In [36]:
from IPython.display import Image, display
from langchain.tools import tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent, ToolNode, ToolRuntime, tools_condition
from langgraph.types import Command
from pydantic import BaseModel, Field
from typing import Annotated, Any, Dict, List, Literal
from typing_extensions import TypedDict
from dotenv import load_dotenv
from pprint import pprint
import os
import random


load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")

In [37]:
PAGE_INDEX = os.getenv("PAGE_INDEX")

In [38]:
from pageindex import PageIndexClient

pi_client = PageIndexClient(api_key=PAGE_INDEX)
pi_client

In [39]:
#document ingestiona dn parsing
from langchain_community.document_loaders import PyPDFLoader
pdfs = [
    "project1/1706.03762v7.pdf",   # Attention Is All You Need (Transformer)
    "project1/2005.11401v4.pdf",   # Retrieval-Augmented Generation (RAG)
    "project1/2210.03629v3.pdf",   # ReAct
    "project1/2310.11511v1.pdf",   # Self-RAG
    "project1/2401.15884v3.pdf",   # Corrective RAG (CRAG)
]


In [40]:
ids = {}

for doc in pdfs:
    result = pi_client.submit_document(doc)
    ids[doc] = result["doc_id"]

ids

C:\Users\Nour_\AppData\Local\Temp\ipykernel_19772\221099963.py:4: UserWarning: Document "1706.03762v7.pdf" was stored as "1706.03762v7_1.pdf".
  result = pi_client.submit_document(doc)
C:\Users\Nour_\AppData\Local\Temp\ipykernel_19772\221099963.py:4: UserWarning: Document "2005.11401v4.pdf" was stored as "2005.11401v4_1.pdf".
  result = pi_client.submit_document(doc)
C:\Users\Nour_\AppData\Local\Temp\ipykernel_19772\221099963.py:4: UserWarning: Document "2210.03629v3.pdf" was stored as "2210.03629v3_1.pdf".
  result = pi_client.submit_document(doc)
C:\Users\Nour_\AppData\Local\Temp\ipykernel_19772\221099963.py:4: UserWarning: Document "2310.11511v1.pdf" was stored as "2310.11511v1_1.pdf".
  result = pi_client.submit_document(doc)
C:\Users\Nour_\AppData\Local\Temp\ipykernel_19772\221099963.py:4: UserWarning: Document "2401.15884v3.pdf" was stored as "2401.15884v3_1.pdf".
  result = pi_client.submit_document(doc)


{'project1/1706.03762v7.pdf': 'pi-cmvvftr6100lw43nqe73vh39a',
 'project1/2005.11401v4.pdf': 'pi-cmvvfts2b00tx0bp6vfr614o8',
 'project1/2210.03629v3.pdf': 'pi-cmvvftssj00lx43nq5635a24c',
 'project1/2310.11511v1.pdf': 'pi-cmvvfttz400ty0bp6tjtqpfwz',
 'project1/2401.15884v3.pdf': 'pi-cmvvftuou00ly43nqx901dtxp'}

In [41]:
ids.keys()

dict_keys(['project1/1706.03762v7.pdf', 'project1/2005.11401v4.pdf', 'project1/2210.03629v3.pdf', 'project1/2310.11511v1.pdf', 'project1/2401.15884v3.pdf'])

In [42]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")
import time

for id in ids.values():
    while True:
        status_result = pi_client.get_document(id)
        status = status_result.get("status")
        print(f"Status : {status}")

        if status =="completed":
            print("\n tree index ready!")
            break
        elif status =="failed":
            print("\nprocessing failed . check yout pdf format")
            break

        time.sleep(5)
    

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
Status : processing
Status : processing
Status : processing
Status : completed

 tree index ready!
Status : processing
Status : processing
Status : processing
Status : completed

 tree index ready!
Status : completed

 tree index ready!
Status : processing
Status : processing
Status : completed

 tree index ready!
Status : completed

 tree index ready!


In [48]:
import json
page_indexes = {}
for id in ids.values():
    tree_result = pi_client.get_tree(id , node_summary=True)
    pageindex_tree = tree_result.get("result", [])
    page_indexes[id] = pageindex_tree

    print(f"📊 Top-level sections: {len(pageindex_tree)}")
    print("\n🌲 Raw tree (first node):")
    print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))




📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 15,
  "summary": "This section, titled \"Attention Is All You Need,\" introduces the Transformer model architecture developed by researchers from Google Brain, Google Research, and the University of Toronto. It covers author metadata, an abstract, and an introduction presenting the Transformer as a novel, highly parallelizable sequence modeling architecture that replaces traditional recurrent and convolutional networks with purely attention-based mechanisms. It provides background on reducing sequential computation, details the encoder-decoder model architecture (including multi-head attention, feed-forward networks, and positional encodings), and analyzes the theoretical advantages of self-attention regarding computational complexity, parallelization, and path length. Furthermore, the section outlines the training regime, presents experime

In [49]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes: list, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("start_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
for doc_id, tree in page_indexes.items():
    print(f"\n=== {doc_id} ===")
    print_tree(tree)


📚 Full Document Structure:


=== pi-cmvvftr6100lw43nqe73vh39a ===
[0000] Attention Is All You Need  (p.1)
  └─ [0001] Attention Is All You Need (intro)  (p.1)
  └─ [0002] Abstract  (p.1)
  └─ [0003] 1 Introduction  (p.2)
  └─ [0004] 2 Background  (p.2)
  └─ [0005] 3 Model Architecture  (p.2)
    └─ [0006] 3 Model Architecture (intro)  (p.2)
    └─ [0007] 3.1 Encoder and Decoder Stacks  (p.3)
    └─ [0008] 3.2 Attention  (p.3)
      └─ [0009] 3.2 Attention (intro)  (p.3)
      └─ [0010] 3.2.1 Scaled Dot-Product Attention  (p.4)
      └─ [0011] 3.2.2 Multi-Head Attention  (p.4)
      └─ [0012] 3.2.3 Applications of Attention in our Model  (p.5)
    └─ [0013] 3.3 Position-wise Feed-Forward Networks  (p.5)
    └─ [0014] 3.4 Embeddings and Softmax  (p.5)
    └─ [0015] 3.5 Positional Encoding  (p.6)
  └─ [0016] 4 Why Self-Attention  (p.6)
  └─ [0017] 5 Training  (p.7)
  └─ [0018] 6 Results  (p.8)
    └─ [0019] 6.1 Machine Translation  (p.8)
    └─ [0020] 6.2 Model Variations  (p.8)
    └─ [0

In [50]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes:list):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

for doc_id, tree in page_indexes.items():
    print(f"\n=== {doc_id} ===")
    total = count_nodes(tree)
    print(f"🔢 Total nodes in tree: {total}")
    print("   Each node = one retrievable section of the document")



=== pi-cmvvftr6100lw43nqe73vh39a ===
🔢 Total nodes in tree: 25
   Each node = one retrievable section of the document

=== pi-cmvvfts2b00tx0bp6vfr614o8 ===
🔢 Total nodes in tree: 32
   Each node = one retrievable section of the document

=== pi-cmvvftssj00lx43nq5635a24c ===
🔢 Total nodes in tree: 30
   Each node = one retrievable section of the document

=== pi-cmvvfttz400ty0bp6tjtqpfwz ===
🔢 Total nodes in tree: 35
   Each node = one retrievable section of the document

=== pi-cmvvftuou00ly43nqx901dtxp ===
🔢 Total nodes in tree: 25
   Each node = one retrievable section of the document


In [51]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, page_indexes: dict) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + a compact outline of every document tree to an LLM.
    LLM reasons over the structure and returns relevant (doc_id, node_id) pairs.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' ([{doc_id, node_id}, ...])
    """
    
    # Short, readable labels (e.g. "1706.03762v7") instead of the long pi-... ids to save tokens
    label_to_doc_id = {path.split("/")[-1].removesuffix(".pdf"): doc_id for path, doc_id in ids.items()}
    doc_id_to_label = {doc_id: label for label, doc_id in label_to_doc_id.items()}
    
    # Plain indented outline (title + page only) — far fewer tokens than indented JSON with summaries
    def outline(nodes: list, depth: int = 0) -> list:
        lines = []
        for n in nodes:
            lines.append(f"{'  ' * depth}{n['node_id']} {n['title']} (p.{n.get('start_index', '?')})")
            if n.get("nodes"):
                lines.extend(outline(n["nodes"], depth + 1))
        return lines
    
    # node_ids repeat across documents (each tree starts at "0000"), so keep a doc label with each outline
    doc_blocks = []
    for doc_id, tree in page_indexes.items():
        about = (tree[0].get("summary", "")[:200].replace("\n", " ") if tree else "")
        doc_blocks.append(f"### DOC {doc_id_to_label.get(doc_id, doc_id)}: {about}\n" + "\n".join(outline(tree)))
    
    prompt = f"""You are given a query and several documents' outlines (like Tables of Contents).
Each line is: <node_id> <section title> (p.<start page>). Indentation shows nesting.
Your task: identify which nodes most likely contain the answer to the query.
Think briefly about which documents and sections are relevant.

Query: {query}

Documents:
{chr(10).join(doc_blocks)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<short reasoning>",
  "node_list": [{{"doc": "<DOC label>", "node_id": "<node_id>"}}]
}}"""

    response = llm.invoke(prompt, response_format={"type": "json_object"})
    result = json.loads(response.content)
    
    # Map the short labels back to the real PageIndex doc_ids
    result["node_list"] = [
        {"doc_id": label_to_doc_id.get(item["doc"], item["doc"]), "node_id": item["node_id"]}
        for item in result.get("node_list", [])
    ]
    return result

In [52]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "what is attention mechanism ?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, page_indexes)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: what is attention mechanism ?

🧠 LLM Reasoning:
The query asks for a definition of the attention mechanism, which is covered in the Transformer paper sections on Attention and its variants.

🎯 Selected Node IDs: [{'doc_id': 'DOC 1706.03762v7', 'node_id': '0008'}, {'doc_id': 'DOC 1706.03762v7', 'node_id': '0009'}, {'doc_id': 'DOC 1706.03762v7', 'node_id': '0010'}, {'doc_id': 'DOC 1706.03762v7', 'node_id': '0011'}, {'doc_id': 'DOC 1706.03762v7', 'node_id': '0016'}]
